In [45]:
from pathlib import Path

from mpi4py import MPI
from petsc4py.PETSc import ScalarType  # type: ignore

import numpy as np

import ufl
from dolfinx import fem, io, mesh, plot
from dolfinx.fem.petsc import LinearProblem

## FEM setup and solution

### Problem parameters

In [46]:
BL_x = 0.0
BL_y = 0.0
TR_x = 2.0
TR_y = 1.0
nx = 4
ny = 4
k = 1
A = 1
q = 1
beta = 0
P = 1
c = beta * P
T1 = 0
T2 = 1
Tinf = 0.5
fdeg = 1
LL = TR_x - BL_x

### Create mesh and find / assign boundaries

In [47]:
msh = mesh.create_rectangle(
    comm=MPI.COMM_WORLD,
    points=((BL_x, BL_y), (TR_x, TR_y)),
    n=(nx, ny),
    cell_type=mesh.CellType.quadrilateral,
)
V = fem.functionspace(msh, ("Lagrange", fdeg))

In [48]:
tdim = msh.topology.dim
fdim = tdim - 1
bnd_left = mesh.locate_entities_boundary(
    msh,
    dim=fdim,
    marker=lambda x: np.isclose(x[0], BL_x),
)
bnd_right = mesh.locate_entities_boundary(
    msh,
    dim=fdim,
    marker=lambda x: np.isclose(x[0], TR_x),
)
dofs_left = fem.locate_dofs_topological(V=V, entity_dim=fdim, entities=bnd_left)
dofs_right = fem.locate_dofs_topological(V=V, entity_dim=fdim, entities=bnd_right)

### Dirichlet boundary conditions

In [49]:
bc_left = fem.dirichletbc(value=ScalarType(T1), dofs=dofs_left, V=V)
bc_right = fem.dirichletbc(value=ScalarType(T2), dofs=dofs_right, V=V)
bcs = [bc_left, bc_right]

### Neumann boundary conditions

In [50]:
g = fem.Constant(msh, ScalarType(0))

### Problem setup

In [51]:
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
x = ufl.SpatialCoordinate(msh)
#f = 10 * ufl.exp(-((x[0] - 0.5) ** 2 + (x[1] - 0.5) ** 2) / 0.02)
f0 = A*q + c*Tinf
f = fem.Constant(msh, ScalarType(f0))
#g = ufl.sin(5 * x[0])
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = ufl.inner(f, v) * ufl.dx + ufl.inner(g, v) * ufl.ds

### Problem solution

In [52]:
problem = LinearProblem(
    a,
    L,
    bcs=bcs,
    petsc_options_prefix="demo_poisson_",
    petsc_options={"ksp_type": "preonly", "pc_type": "lu", "ksp_error_if_not_converged": True},
)
uh = problem.solve()
assert isinstance(uh, fem.Function)
uh.name = "T_FEM"

In [53]:
out_folder = Path("dirichlet")
out_folder.mkdir(parents=True, exist_ok=True)
with io.XDMFFile(msh.comm, out_folder / "dirichlet.xdmf", "w") as file:
    file.write_mesh(msh)
    file.write_function(uh)

# Analytical solution
The following is the code to compute the analytical solution.
The solution has been taken from "An Introduction to the Finite Element Method" by J.N.Reddy

In [54]:
mshAnalytical = mesh.create_rectangle(
    comm=MPI.COMM_WORLD,
    points=((BL_x, BL_y), (TR_x, TR_y)),
    n=(100, 100),
    cell_type=mesh.CellType.quadrilateral,
)
A = fem.functionspace(mshAnalytical, ("Lagrange", 1))

In [55]:
def analytical_T(x):
    return q*LL**2 / (2*k) * (x[0]/LL - (x[0]/LL)**2) + (T2-T1)*(x[0]/LL) + T1

In [56]:
u_analytical = fem.Function(A)
u_analytical.name = "T_analytical"
u_analytical.interpolate(analytical_T)

In [57]:
with io.XDMFFile(mshAnalytical.comm, out_folder / "primary_analytical.xdmf", "w") as file:
    file.write_mesh(mshAnalytical)
    file.write_function(u_analytical)